# IPC2BNS-Verify — Phase 6: Master Evaluation, Ablation Summary & Final Report

This master notebook runs the complete end-to-end evaluation harness across all 4 stages:
1. **Stage 1 (Baseline LLM, Closed-Book)**
2. **Stage 2 (+RAG Statutory Context)**
3. **Stage 3 (+Two-Layer Hard-Constraint Verifier)**
4. **Stage 4 (+Verifier + Incremental Refresh)**
5. **Master Ablation Summary Table Generation** (`ablation_summary_table.csv`)
6. **Full 65-Test Automated Pytest Suite Execution**

---
## 1. Mount Google Drive & Environment Setup

In [1]:
from google.colab import drive
import os, sys, shutil

# Mount Drive cleanly
drive.mount('/content/drive', force_remount=False)

DRIVE_ROOT = '/content/drive/MyDrive/NLP_rspaper'
LOCAL_ROOT = '/content/IPC2BNS-Verify'

# Copy to Colab local SSD for lightning-fast disk I/O & zero network timeouts
if os.path.exists(DRIVE_ROOT):
    if os.path.exists(LOCAL_ROOT):
        shutil.rmtree(LOCAL_ROOT)
    shutil.copytree(DRIVE_ROOT, LOCAL_ROOT, ignore=shutil.ignore_patterns('__pycache__', '.pytest_cache', '.git'))
    PROJECT_ROOT = LOCAL_ROOT
    print('✅ Synced project from Drive to Colab local SSD:', PROJECT_ROOT)
else:
    PROJECT_ROOT = DRIVE_ROOT

os.environ['IPC2BNS_PROJECT_ROOT'] = PROJECT_ROOT
if os.path.join(PROJECT_ROOT, 'code') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_ROOT, 'code'))

print('Environment initialized.')

Mounted at /content/drive
✅ Synced project from Drive to Colab local SSD: /content/IPC2BNS-Verify
Environment initialized.


---
## 2. Dependencies

In [2]:
!pip install -q pytest pandas tabulate
print('Dependencies ready.')

Dependencies ready.


---
## 3. Run Master Evaluation Harness Across All 4 Stages

In [3]:
from src.eval.harness import MasterEvaluationHarness, generate_full_ablation_report

results_dir = os.path.join(PROJECT_ROOT, 'results')
out_csv = os.path.join(results_dir, 'ablation_summary_table.csv')

harness = MasterEvaluationHarness(results_dir)
ablation_rows = harness.export_ablation_summary_csv(out_csv)

import pandas as pd
df = pd.DataFrame(ablation_rows)
print('\n' + '='*85)
print('MASTER ABLATION SUMMARY TABLE')
print('='*85)
display(df)


MASTER ABLATION SUMMARY TABLE


,stage_id,system_configuration,sample_size_N,citation_accuracy,confidence_interval_95,hallucination_catch_rate,false_positive_rate,amendment_adaptivity_delta,statutory_reliability_score
0,Stage 1,Baseline LLM (Closed-Book),N=60 dev queries,10.0% (6/60),[4.7% - 20.1%],N/A (No Verifier),N/A,N/A,5.0%
1,Stage 2,+BM25 RAG (Retrieved Context),N=60 dev queries,63.3% (38/60),[50.7% - 74.4%],N/A (No Verifier),N/A,N/A,53.8%
2,Stage 3,+Two-Layer Hard Verifier,N=30 stress cases (18 adv + 12 ctrl),63.3% (38/60),[50.7% - 74.4%],100.0% (18/18 caught) [82.4%-100.0%],0.0% (0/12 rejected),33.3% (1/3 pre-refresh),95.0%
3,Stage 4,+Incremental Refresh (Full System),N=3 amended provision queries,63.3% (38/60),[50.7% - 74.4%],100.0% (18/18 caught),0.0% (0/12 rejected),+66.7% delta (1/3 -> 3/3) [43.8%-100.0%],98.5%
4,Generalization,CrPC (1973) <-> BNSS (2023) Procedural Set,N=25 procedural queries,100.0% (25/25),[86.7% - 100.0%],100.0% (Caught Remand/Bail drift),0.0%,N/A (Static Code Pair),98.0%


---
## 4. Human Expert Calibration Inspection

In [4]:
human_cal_file = os.path.join(PROJECT_ROOT, 'results/human_review_calibration.csv')
cal_df = pd.read_csv(human_cal_file)
print('=== DOUBLE-BLIND LEGAL EXPERT CALIBRATION (SAMPLE) ===')
display(cal_df[['question_id', 'consensus_verdict', 'inter_annotator_agreement_cohen_kappa', 'verifier_alignment_status']])

=== DOUBLE-BLIND LEGAL EXPERT CALIBRATION (SAMPLE) ===


,question_id,consensus_verdict,inter_annotator_agreement_cohen_kappa,verifier_alignment_status
0,DEV_001,ACCURATE,0.92,PERFECT_ALIGNMENT
1,DEV_002,ACCURATE,0.95,PERFECT_ALIGNMENT
2,DEV_006,ACCURATE,0.88,PERFECT_ALIGNMENT
3,DEV_010,AUTHORITATIVE_VETO,0.96,PERFECT_ALIGNMENT
4,DEV_011,AUTHORITATIVE_VETO,0.98,PERFECT_ALIGNMENT
5,DEV_013,ACCURATE,0.94,PERFECT_ALIGNMENT
6,DEV_014,ACCURATE,0.90,PERFECT_ALIGNMENT


---
## 5. Qualitative Error Analysis Notes

In [5]:
error_notes_file = os.path.join(PROJECT_ROOT, 'results/error_analysis_notes.md')
with open(error_notes_file, 'r', encoding='utf-8') as f:
    print(f.read())

# Error Analysis & Qualitative Breakdown — IPC2BNS-Verify

## 1. Executive Summary

This document details the systematic error analysis across the 4 experimental ablation stages of **IPC2BNS-Verify** over Indian Penal Code (IPC 1860) to Bharatiya Nyaya Sanhita (BNS 2023) statutory transition questions.

---

## 2. Stage-by-Stage Failure Modes

### Stage 1: Closed-Book Baseline LLM (No Retrieval)
- **Failure Mode 1: Historical Inertia & Parametric Memory Bias**
  - The model frequently defaults to pre-trained IPC section numbers (e.g. citing `[IPC §420]` for cheating or `[IPC §302]` for murder) even when the query explicitly asks for BNS 2023.
  - *Accuracy:* Only **35.3%**.
- **Failure Mode 2: Inability to Capture Granular Statutory Sub-Clauses**
  - For new criminal provisions like hit-and-run driving penalties under BNS §106(2) or mob lynching under BNS §103(2), the baseline model generates generic statements with wrong or missing section numbers.

### Stage 2: +RAG (Statutory Contex

---
## 6. Run Complete Automated Pytest Suite (All 65 Tests)

In [6]:
test_dir = os.path.join(PROJECT_ROOT, 'code/tests')
!python -m pytest "{test_dir}" -v --color=yes

============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0 -- /usr/bin/python3
cachedir: .pytest_cache
rootdir: /content
plugins: typeguard-4.6.0, langsmith-0.11.1, anyio-4.14.2
collected 67 items                                                             

IPC2BNS-Verify/code/tests/test_concordance.py::test_concordance_table_loads_successfully PASSED [  1%]
IPC2BNS-Verify/code/tests/test_concordance.py::test_concordance_schema_columns PASSED [  2%]
IPC2BNS-Verify/code/tests/test_concordance.py::test_deterministic_exact_mappings[302-103] PASSED [  4%]
IPC2BNS-Verify/code/tests/test_concordance.py::test_deterministic_exact_mappings[299-100] PASSED [  5%]
IPC2BNS-Verify/code/tests/test_concordance.py::test_deterministic_exact_mappings[304A-106] PASSED [  7%]
IPC2BNS-Verify/code/tests/test_concordance.py::test_deterministic_exact_mappings[304B-80] PASSED [  8%]
IPC2BNS-Verify/code/tests/test_concordance.py

---
## 7. Check Final WBS Project Completion

In [7]:
!python "{PROJECT_ROOT}/check_progress.py" --root "{PROJECT_ROOT}" --write-report

# Project Progress Report
**Overall: 32/32 tasks complete (100%)**

_Generated: 2026-09-03T09:50:06_

## 0. Setup — 4/4 (100%)
- [x] Repo scaffolding + config system  `(code/src, code/configs)`
- [x] India Code raw text downloaded  `(data/00_raw/india_code)`
- [x] Concordance source PDF(s) collected  `(data/00_raw/concordance_source_pdfs)`
- [x] Data Management Plan written  `(docs/IPC2BNS-Verify_Data_Management_Plan.md)`

## 1. Mapping Module — 5/5 (100%)
- [x] Ground-truth concordance table finalized  `(data/02_ground_truth/concordance_v1.csv)`
- [x] Concordance validation report reviewed  `(data/02_ground_truth/validation_report.csv)`
- [x] Deterministic lookup function implemented  `(code/src/mapping/lookup.py)`
- [x] Query normalizer implemented  `(code/src/mapping/normalizer.py)`
- [x] Mapping module unit tests  `(code/tests/test_concordance.py)`

## 2. Ingestion & Retrieval — 6/6 (100%)
- [x] Section-level chunker implemented  `(code/src/ingestion/chunker.py)`
- [x] Cleaned sect

---
## 8. Sync Results Back to Google Drive

In [8]:
if PROJECT_ROOT == LOCAL_ROOT:
    import shutil
    shutil.copy2(os.path.join(PROJECT_ROOT, 'results/ablation_summary_table.csv'), os.path.join(DRIVE_ROOT, 'results/ablation_summary_table.csv'))
    shutil.copy2(os.path.join(PROJECT_ROOT, 'results/progress_report.md'), os.path.join(DRIVE_ROOT, 'results/progress_report.md'))
    print('✅ Saved latest results to Google Drive successfully.')

✅ Saved latest results to Google Drive successfully.
